# HW 03

## 1a
Read up on network configuration model!

## 1b


In [ ]:
import numpy as np
import networkx as nx

def configuration_model(degrees, return_graph = False):
    
    # Molloy–Reed configuration model.
    # degrees: list/array of nonnegative integers, node i has degree degrees[i]
    # return_graph=False -> return edge list; True -> return nx.MultiGraph
    
    # basic checks
    deg = list(degrees)
    if any((d < 0) or (int(d) != d) for d in deg):
        raise ValueError("Degrees must be nonnegative integers")
    if sum(deg) % 2 != 0:
        raise ValueError("Sum of degrees must be even")

    # expand degrees into a stub list of node ids
    stubs = []
    for i, d in enumerate(deg):
        for _ in range(int(d)):
            stubs.append(i)

    # random permutation and sequential pairing
    stubs = np.array(stubs, dtype = int)
    np.random.shuffle(stubs)
    edges = []
    for i in range(0, len(stubs), 2):
        u = int(stubs[i])
        v = int(stubs[i + 1])
        edges.append((u, v))

    # graph return
    if return_graph:
        G = nx.MultiGraph()
        G.add_nodes_from(range(len(deg)))  # keep isolated nodes
        G.add_edges_from(edges)
        return G
    else:
        return edges


In [ ]:
deg = [3, 2, 1, 0, 2, 2]
E = configuration_model(deg)                        # edge list
G = configuration_model(deg, return_graph = True)   # MultiGraph


# 1c

In [12]:
import numpy as np
import pandas as pd
import networkx as nx

# guide code verbatim
def power_law(n: int, minval: int, maxval: int, alpha: float, seed: int | None = None) -> np.ndarray:
    if seed is not None:
        np.random.seed(seed)
    u = np.random.random(n)
    if alpha == -1:
        return np.round(np.exp(np.log(minval - 0.5) + u * (np.log(maxval + 0.5) - np.log(minval - 0.5)))).astype(int)
    else:
        a = (minval - 0.5) ** (1 + alpha)
        b = (maxval + 0.5) ** (1 + alpha)
        return np.round((a + u * (b - a)) ** (1 / (1 + alpha))).astype(int)

# plain configuration model: expand stubs -> shuffle -> pair
def configuration_model(degrees, return_graph=False):
    d = list(degrees)
    stubs = []
    for i, k in enumerate(d):
        for _ in range(int(k)):
            stubs.append(i)
    stubs = np.array(stubs, dtype=int)
    np.random.shuffle(stubs)
    edges = []
    for i in range(0, len(stubs), 2):
        u = int(stubs[i])
        v = int(stubs[i+1])
        edges.append((u, v))
    if return_graph:
        G = nx.MultiGraph()
        G.add_nodes_from(range(len(d)))
        G.add_edges_from(edges)
        return G
    return edges

# parameters 
N = 1000
minval = 1
maxval = N - 1
alpha = -2.5

# one degree sequence
deg = power_law(N, minval, maxval, alpha, seed=42)
if deg.sum() % 2 == 1:                       # make even by adding 1 to a random node
    deg[np.random.randint(0, N)] += 1

# 1000 realizations: fractions of self-loops and multi-edges
T = 1000
loop_frac = []
multi_frac = []

for _ in range(T):
    E = configuration_model(deg)             # edge list
    m = len(E)

    # self-loops
    loops = 0
    for u, v in E:
        if u == v:
            loops += 1

    # multi-edges beyond one copy per unordered pair
    counts = {}
    for u, v in E:
        if u <= v:
            key = (u, v)
        else:
            key = (v, u)
        if key in counts:
            counts[key] += 1
        else:
            counts[key] = 1
    multi_extra = 0
    for key in counts:
        c = counts[key]
        if c > 1:
            multi_extra += (c - 1)

    loop_frac.append(loops / m)
    multi_frac.append(multi_extra / m)

print("Configuration model over 1000 runs")
print("avg self-loop fraction:", float(np.mean(loop_frac)))
print("avg multi-edge fraction:", float(np.mean(multi_frac)))

# compare to budapest_connectome 
df = pd.read_csv("../../data/buda-edge-list.csv", sep=";")
edges_b = list(zip(df["id node1"].astype(int), df["id node2"].astype(int)))
m_b = len(edges_b)

loops_b = 0
for u, v in edges_b:
    if u == v:
        loops_b += 1

counts_b = {}
for u, v in edges_b:
    if u <= v:
        key = (u, v)
    else:
        key = (v, u)
    if key in counts_b:
        counts_b[key] += 1
    else:
        counts_b[key] = 1

multi_extra_b = 0
for key in counts_b:
    c = counts_b[key]
    if c > 1:
        multi_extra_b += (c - 1)

print("\nbudapest_connectome")
print("self-loop fraction:", loops_b / m_b)
print("multi-edge fraction:", multi_extra_b / m_b)


Configuration model over 1000 runs
avg self-loop fraction: 0.006315989847715736
avg multi-edge fraction: 0.013234771573604061

budapest_connectome
self-loop fraction: 0.011
multi-edge fraction: 0.0
